In [150]:
import warnings
warnings.simplefilter(action='ignore')

import os
import requests
import joblib

import numpy
import pandas

import matplotlib.pyplot as plt
import scipy.stats as stats

In [151]:
from catboost import CatBoostRegressor ,Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV, LassoCV, RidgeCV
from sklearn.svm import SVR

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import accuracy_score

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import KFold, cross_val_score

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import SelectFromModel

from sklearn.impute import SimpleImputer
import sklearn.linear_model as linear_model
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

In [152]:
train = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/train.csv')
test = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv').drop(columns=['id'])

In [153]:
encoder = LabelEncoder()
normalize = SimpleImputer(strategy='mean')

In [154]:
for i in zip(train.columns,train.dtypes):
    if i[1]=='O':
        train[i[0]] = train[i[0]].fillna('Unknown')
        train[i[0]]=normalize.fit_transform(encoder.fit_transform(train[i[0]].to_numpy().reshape(-1,1)).reshape(-1,1))
    else:
        train[i[0]].fillna(train[i[0]].mean(), inplace=True)
        train[i[0]]=normalize.fit_transform(numpy.array(train[i[0]]).reshape(-1,1))
for i in zip(test.columns,test.dtypes):
    if i[1]=='O':
        test[i[0]] = test[i[0]].fillna('Unknown')
        test[i[0]]=normalize.fit_transform(numpy.array(encoder.fit_transform(test[i[0]].to_numpy().reshape(-1,1))).reshape(-1,1))
    else:
        test[i[0]].fillna(test[i[0]].mean(), inplace=True)
        test[i[0]]=normalize.fit_transform(numpy.array(test[i[0]]).reshape(-1,1))

In [155]:
train_x = train.drop(columns=['id', 'efficiency'])

train_y = train['efficiency']

In [156]:
X_train, X_test, y_train, y_test = train_test_split(train_x, train_y, test_size=0.1, random_state=100)

In [157]:
catboost = CatBoostRegressor(grow_policy='Depthwise',
                             iterations=3000,
                             eval_metric='RMSE',
                             random_state=0,
                             verbose=100)

cat_features = list(train_x.select_dtypes(include=['object', 'category']).columns)
    
train_pool = Pool(X_train, y_train, cat_features=cat_features)
val_pool = Pool(X_test, y_test, cat_features=cat_features)
        
catboost.fit(train_pool, eval_set=(val_pool), verbose=100, early_stopping_rounds=100)

Learning rate set to 0.041014
0:	learn: 0.1387230	test: 0.1331053	best: 0.1331053 (0)	total: 17.2ms	remaining: 51.5s
100:	learn: 0.0999202	test: 0.0977807	best: 0.0977807 (100)	total: 1.4s	remaining: 40.1s
200:	learn: 0.0960941	test: 0.0978478	best: 0.0977215 (152)	total: 2.76s	remaining: 38.4s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.09772146713
bestIteration = 152

Shrink model to first 153 iterations.


In [162]:
cat_pre_test = catboost.predict(test)
print(f'Concordance index (lifelines): {mean_absolute_error(y_test, catboost.predict(X_test))}')

Concordance index (lifelines): 0.05117214886497807


In [163]:
id = pandas.read_csv('/kaggle/input/zelestra-x-aws-ml-ascend-challenge/test.csv')['id']
test_predictions = (cat_pre_test)
test_predictions

array([0.40372314, 0.53627379, 0.51449115, ..., 0.61762357, 0.43276002,
       0.56316936])

In [164]:
submission = pandas.DataFrame({
    'id': id.values,
    'efficiency': test_predictions,
})
submission

,id,efficiency
0,0,0.403723
1,1,0.536274
2,2,0.514491
3,3,0.462554
4,4,0.456636
...,...,...
11995,11995,0.555241
11996,11996,0.460051
11997,11997,0.617624
11998,11998,0.432760


In [161]:
submission.to_csv('submission.csv', index = False)